# Lab 29 — A2A endpoint at production depth

> ⏱ 110-130 min · 🟡 Intermediate

Extend Lab 28's in-memory A2A server with five production concerns end-to-end: `DatabaseTaskStore` with SQLite (persistence across restart), JWS-signed Agent Card (cryptographic identity verification), API-key middleware (Starlette `BaseHTTPMiddleware`), streaming `SendStreamingMessage` (SSE), and OpenTelemetry tracing (captured to a JSON-line file for inspection).

**Prerequisites**: [Lab 28](../28-a2a-endpoint-from-scratch/) (you'll extend its `hello_agent_server.py` pattern), [Module 6](../../concepts/tools/a2a-endpoint-production-depth.md).

**Strategy**: same subprocess pattern as Lab 28 — uvicorn runs in its own process, the notebook drives via httpx. Two subprocess lifetimes (Steps 3-5, then Steps 6-7-8) demonstrate task persistence across server restart by inspecting the SQLite DB and OTel spans file directly between them.

## Step 0 — Environment setup

Lab 29 needs more dependencies than Lab 28: `sqlalchemy[asyncio]` + `aiosqlite` for the database, `joserfc` for JWS signing, `opentelemetry-sdk` for tracing. All open-source, no external services.

In [ ]:
import sys
import subprocess
import time
import json
import base64
import sqlite3
from pathlib import Path

LAB_DIR = Path.cwd()
print(f"Lab working directory: {LAB_DIR}")

# Required imports
import_failures = []
for mod_name, install_hint in [
    ("a2a", "pip install 'a2a-sdk>=1.0,<2.0'"),
    ("sqlalchemy", "pip install 'sqlalchemy>=2.0'"),
    ("aiosqlite", "pip install aiosqlite"),
    ("joserfc", "pip install joserfc"),
    ("opentelemetry", "pip install 'opentelemetry-sdk>=1.20'"),
    ("httpx", "pip install httpx"),
    ("uvicorn", "pip install uvicorn"),
]:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name}")
    except ImportError:
        import_failures.append((mod_name, install_hint))
        print(f"  ✗ {mod_name} — {install_hint}")

if import_failures:
    print("\nFix the failures above and re-run this cell.")
    sys.exit(1)

print("\n✓ Environment ready. No API keys or external services required.")

## Step 1 — Generate RSA keypair + sign the Agent Card via JWS

Module 6 explains the JWS-signed Agent Card pattern: an agent's domain owner signs the card with a private key; receiving agents verify with the corresponding public key. The protocol shape follows [RFC 7515 (JWS)](https://datatracker.ietf.org/doc/html/rfc7515).

Three things this cell does:

1. **Generate an RSA-2048 keypair** via `joserfc.jwk.RSAKey.generate_key`. Why RSA-2048 and not Ed25519: bare `EdDSA` was deprecated by RFC 9864 and joserfc raises a `SecurityWarning`. RS256 is universally supported, not subject to algorithm-name churn, and well-understood by every auth stack. For a production deployment that wants Ed25519, use the `Ed25519` algorithm name explicitly per RFC 9864.
2. **Build the unsigned Agent Card** with `streaming: true` capability declared (so Step 7's `SendStreamingMessage` works) and an `apiKey` security scheme declared (so Step 4's auth enforcement is also declared in the card).
3. **Sign + attach** — canonicalize the card JSON via `MessageToDict` + `json.dumps(sort_keys=True)`, JWS-RS256-sign with the private key, attach the resulting `AgentCardSignature` to the card's `signatures` field, save card + public key to disk.

In [ ]:
from joserfc import jws
from joserfc.jws import JWSRegistry
from joserfc.jwk import RSAKey
from google.protobuf.json_format import MessageToDict
from a2a.types import (
    AgentCard, AgentSkill, AgentCapabilities, AgentInterface,
    SecurityScheme, APIKeySecurityScheme, AgentCardSignature,
)

# 1. Generate the agent's RSA-2048 keypair (the agent's private key stays with the agent)
key = RSAKey.generate_key(2048)
print("✓ Generated RSA-2048 keypair")

# 2. Build the unsigned Agent Card as protobuf
card = AgentCard(
    name="production-echo-agent",
    description="Lab 29 production-shape A2A endpoint",
    version="1.0.0",
    capabilities=AgentCapabilities(
        streaming=True,           # required for Step 7
        push_notifications=False,  # out of scope for Lab 29
    ),
    supported_interfaces=[
        AgentInterface(protocol_binding="JSONRPC", url="http://127.0.0.1:9998")
    ],
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    skills=[
        AgentSkill(
            id="echo", name="Echo", description="Echo input back",
            tags=["echo"],
            input_modes=["text/plain"], output_modes=["text/plain"],
        )
    ],
    security_schemes={
        "apiKey": SecurityScheme(
            api_key_security_scheme=APIKeySecurityScheme(
                location="header", name="X-API-Key",
            )
        )
    },
)

# 3. Canonicalize and sign
card_dict = MessageToDict(card)
payload_bytes = json.dumps(card_dict, sort_keys=True).encode()
registry = JWSRegistry(algorithms=["RS256"])
signed_compact = jws.serialize_compact(
    {"alg": "RS256", "typ": "JWS"}, payload_bytes, key, registry=registry,
)

# JWS compact form: <protected_header>.<payload>.<signature>
header_b64, _, sig_b64 = signed_compact.split(".")

# Attach signature to the card via AgentCardSignature proto
card.signatures.append(AgentCardSignature(protected=header_b64, signature=sig_b64))

# Save the signed card and the public key to disk
CARD_PATH = LAB_DIR / "signed_card.json"
PUBKEY_PATH = LAB_DIR / "pub_key.json"

CARD_PATH.write_text(json.dumps(MessageToDict(card), indent=2))
PUBKEY_PATH.write_text(json.dumps(key.as_dict(private=False)))

print(f"\n✓ Wrote {CARD_PATH.name} (size: {CARD_PATH.stat().st_size} bytes)")
print(f"✓ Wrote {PUBKEY_PATH.name} (RSA public key only)")
print(f"\nSignature attached: protected={header_b64}")
print(f"  signature (first 60 chars): {sig_b64[:60]}...")

## Step 2 — Author `production_agent_server.py`

The server extends Lab 28's `hello_agent_server.py` with three production layers:

- **`DatabaseTaskStore`** wired to a SQLite file-backed SQLAlchemy AsyncEngine
- **`APIKeyMiddleware`** — Starlette `BaseHTTPMiddleware` that enforces `X-API-Key` on every request except the public Agent Card discovery URL
- **OpenTelemetry `SimpleSpanProcessor`** writing spans as JSON lines to a file the notebook reads in Step 8

The server loads the signed Agent Card from disk (written by Step 1). The `AgentExecutor.execute()` method is identical to Lab 28's — what changes is the wrapping (persistence + auth + tracing).

In [ ]:
SERVER_PATH = LAB_DIR / "production_agent_server.py"
DB_PATH = LAB_DIR / "a2a_tasks.db"
SPANS_PATH = LAB_DIR / "a2a_spans.jsonl"

SERVER_CODE = """\
\"\"\"Lab 29 production A2A server — DatabaseTaskStore + signed card + APIKey + OTel.\"\"\"
import os
import json
import asyncio
from pathlib import Path

# Configure OTel BEFORE importing a2a so @trace_class picks up the provider
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry.sdk.resources import Resource

SPANS_PATH = os.environ.get(\"A2A_OTEL_SPANS\", \"./a2a_spans.jsonl\")


class JSONLineSpanExporter(SpanExporter):
    \"\"\"Single-line JSON span exporter for notebook inspection.\"\"\"

    def __init__(self, path: str):
        self._path = path

    def export(self, spans):
        with open(self._path, \"a\") as f:
            for span in spans:
                f.write(json.dumps({
                    \"name\": span.name,
                    \"kind\": span.kind.name,
                    \"trace_id\": format(span.context.trace_id, \"032x\"),
                    \"span_id\": format(span.context.span_id, \"016x\"),
                    \"parent_span_id\": format(span.parent.span_id, \"016x\") if span.parent else None,
                    \"duration_us\": (span.end_time - span.start_time) // 1000,
                }) + \"\\n\")
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass


provider = TracerProvider(resource=Resource.create({\"service.name\": \"production-echo-agent\"}))
provider.add_span_processor(SimpleSpanProcessor(JSONLineSpanExporter(SPANS_PATH)))
trace.set_tracer_provider(provider)


# Now safe to import a2a — OTel is configured
from sqlalchemy.ext.asyncio import create_async_engine
from a2a.types import AgentCard
from a2a.server.agent_execution import AgentExecutor
from a2a.server.tasks import DatabaseTaskStore, TaskUpdater
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.helpers import new_text_part, new_task_from_user_message
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse
import uvicorn


API_KEY = os.environ.get(\"A2A_API_KEY\", \"test-key-123\")
DB_URL = os.environ.get(\"A2A_DB_URL\", \"sqlite+aiosqlite:///./a2a_tasks.db\")
CARD_PATH = Path(os.environ.get(\"A2A_CARD_PATH\", \"./signed_card.json\"))


class APIKeyMiddleware(BaseHTTPMiddleware):
    \"\"\"Enforce X-API-Key on RPC; keep .well-known/ public.\"\"\"

    async def dispatch(self, request, call_next):
        if request.url.path.startswith(\"/.well-known/\"):
            return await call_next(request)
        if request.headers.get(\"X-API-Key\") != API_KEY:
            return JSONResponse(
                {\"error\": {\"code\": -32001, \"message\": \"Unauthorized\"}},
                status_code=401,
            )
        return await call_next(request)


class EchoAgent(AgentExecutor):
    async def execute(self, context, event_queue):
        message_text = \"\"
        if context.message and context.message.parts:
            for part in context.message.parts:
                if part.HasField(\"text\"):
                    message_text += part.text

        task = new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()
        await updater.add_artifact(
            parts=[new_text_part(f\"Echo: {message_text}\")],
            name=\"response\",
        )
        await updater.complete()

    async def cancel(self, context, event_queue):
        raise NotImplementedError


def load_signed_card() -> AgentCard:
    from google.protobuf.json_format import ParseDict
    data = json.loads(CARD_PATH.read_text())
    card = AgentCard()
    ParseDict(data, card)
    return card


async def init_engine():
    engine = create_async_engine(DB_URL)
    store = DatabaseTaskStore(engine=engine)
    await store.initialize()
    return engine


def build_app(engine):
    card = load_signed_card()
    handler = DefaultRequestHandler(
        agent_executor=EchoAgent(),
        task_store=DatabaseTaskStore(engine=engine),
        agent_card=card,
    )
    return Starlette(
        routes=create_agent_card_routes(agent_card=card) +
               create_jsonrpc_routes(request_handler=handler, rpc_url=\"/\"),
        middleware=[Middleware(APIKeyMiddleware)],
    )


if __name__ == \"__main__\":
    engine = asyncio.run(init_engine())
    app = build_app(engine)
    uvicorn.run(app, host=\"127.0.0.1\", port=9998, log_level=\"warning\")
"""

SERVER_PATH.write_text(SERVER_CODE)
print(f"Wrote {SERVER_PATH} ({SERVER_PATH.stat().st_size} bytes)")
print(f"\nServer will read from: {CARD_PATH}")
print("Server will write to:")
print(f"  - {DB_PATH} (SQLite task store)")
print(f"  - {SPANS_PATH} (OTel spans as JSON lines)")

## Step 3 — Spawn server; verify JWS signature; demonstrate tamper rejection

Spawn the server subprocess and verify the signed Agent Card end-to-end. Four sub-steps:

3a. **Fetch the card** — the `.well-known/` URL is public; no API key needed.
3b. **Reconstruct the canonical payload + verify the signature** — this requires a subtle step: the SDK's `create_agent_card_routes()` injects three backward-compat fields into the served JSON that aren't in the protobuf schema (`preferredTransport`, `protocolVersion`, `url`). The signature was computed over the protobuf-canonical form. To verify, we round-trip the fetched JSON through `ParseDict(..., ignore_unknown_fields=True)` to drop those injected fields, then re-emit via `MessageToDict()` to get the same canonical bytes the signer saw. This is a real production gotcha — the JWS payload must be byte-identical to what was signed; transport-injected fields must be stripped on the verifier side. ([RFC 7515](https://datatracker.ietf.org/doc/html/rfc7515) discusses signed/unsigned payload boundaries in §7.2.)
3c. **Demonstrate tamper rejection** — mutate one byte of the signature; observe `BadSignatureError`. Then mutate the payload itself; observe rejection again.

In [ ]:
# Clean any leftover state from prior runs
for p in [DB_PATH, SPANS_PATH]:
    if p.exists():
        p.unlink()

import httpx

server_process = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(2.0)

# Health probe
for _attempt in range(5):
    try:
        r = httpx.get("http://127.0.0.1:9998/.well-known/agent-card.json", timeout=2.0)
        if r.status_code == 200:
            break
    except (httpx.ConnectError, httpx.ReadTimeout):
        time.sleep(0.5)
else:
    if server_process.poll() is not None:
        print("Server died:")
        print(server_process.stderr.read().decode()[:1500])
    raise RuntimeError("Server failed to start")

print(f"✓ Server running (PID {server_process.pid}) on port 9998")

# 3a — fetch the card
card_response = httpx.get("http://127.0.0.1:9998/.well-known/agent-card.json", timeout=5.0)
fetched_card = card_response.json()
print(f"\n✓ Fetched card: {fetched_card['name']} v{fetched_card['version']}")
print(f"  Signatures attached: {len(fetched_card.get('signatures', []))}")
print(f"  Streaming declared: {fetched_card['capabilities']['streaming']}")
print(f"  Security schemes: {list(fetched_card.get('securitySchemes', {}).keys())}")

In [ ]:
# 3b — Verify the JWS signature against the public key from Step 1

from google.protobuf.json_format import ParseDict

# Reload the public key from disk
pub_key_data = json.loads(PUBKEY_PATH.read_text())
pub_key = RSAKey.import_key(pub_key_data)

# Extract the signature from the fetched card
sig_obj = fetched_card["signatures"][0]
header_b64 = sig_obj["protected"]
sig_b64 = sig_obj["signature"]

# Reconstruct the canonical payload. Two transformations are necessary:
# (1) strip the 'signatures' field — the JWS payload doesn't include itself.
# (2) round-trip through the AgentCard protobuf with ignore_unknown_fields=True,
#     which drops the SDK-injected v0.3 backward-compat fields (preferredTransport,
#     protocolVersion, url) that aren't in the protobuf schema. The signer canonicalized
#     against the protobuf form; the verifier must produce the same byte sequence.
fetched_no_sigs = {k: v for k, v in fetched_card.items() if k != "signatures"}
card_msg = AgentCard()
ParseDict(fetched_no_sigs, card_msg, ignore_unknown_fields=True)
canonical_dict = MessageToDict(card_msg)
payload_bytes = json.dumps(canonical_dict, sort_keys=True).encode()
payload_b64 = base64.urlsafe_b64encode(payload_bytes).rstrip(b"=").decode()

# Assemble the JWS compact form and verify
compact_jws = f"{header_b64}.{payload_b64}.{sig_b64}"
registry = JWSRegistry(algorithms=["RS256"])

try:
    obj = jws.deserialize_compact(compact_jws, pub_key, registry=registry)
    print("✓ Signature verified — card is authentic and untampered")
    print(f"  Header: {json.loads(base64.urlsafe_b64decode(header_b64 + '==').decode())}")
    print(f"  Payload length: {len(obj.payload)} bytes")
    print("  SDK-injected fields stripped during canonicalization:")
    sdk_only = set(fetched_card.keys()) - set(canonical_dict.keys()) - {"signatures"}
    for f in sdk_only:
        print(f"    - {f}: {fetched_card[f]}")
except Exception as e:
    print(f"✗ Signature verification FAILED: {type(e).__name__}: {e}")

In [ ]:
# 3c — Demonstrate tamper rejection

# Mutate the signature (replace the last 5 chars)
tampered_sig = sig_b64[:-5] + "XXXXX"
tampered_compact = f"{header_b64}.{payload_b64}.{tampered_sig}"

try:
    jws.deserialize_compact(tampered_compact, pub_key, registry=registry)
    print("✗ ALARM: tampered signature was NOT rejected — this should never happen")
except Exception as e:
    print(f"✓ Tamper detected: {type(e).__name__}: {str(e)[:80] or '(empty message)'}")

# Also demonstrate that mutating the payload (not the signature) fails verification
tampered_canonical = {**canonical_dict, "name": "evil-impersonator"}
tampered_payload_b64 = base64.urlsafe_b64encode(
    json.dumps(tampered_canonical, sort_keys=True).encode()
).rstrip(b"=").decode()
tampered_compact_2 = f"{header_b64}.{tampered_payload_b64}.{sig_b64}"

try:
    jws.deserialize_compact(tampered_compact_2, pub_key, registry=registry)
    print("✗ ALARM: tampered payload was NOT rejected")
except Exception as e:
    print(f"✓ Payload tamper detected: {type(e).__name__}")

## Step 4 — API-key authentication

The card declares `apiKey` as a security scheme; the server's `APIKeyMiddleware` enforces it. Three checks:

4a. **Request without API key** — expect HTTP 401, with the JSON-RPC error code `-32001`.
4b. **Request with correct API key** — expect HTTP 200, full Task lifecycle in the response.
4c. **Two more requests** to populate the SQLite database for Steps 5 and 6.

In [ ]:
# 4a — No API key
no_auth_response = httpx.post(
    "http://127.0.0.1:9998/",
    headers={"Content-Type": "application/json", "A2A-Version": "1.0"},
    json={
        "jsonrpc": "2.0", "id": "r0", "method": "SendMessage",
        "params": {"message": {
            "message_id": "m0", "role": "ROLE_USER",
            "parts": [{"text": "hello?"}],
        }},
    },
    timeout=5.0,
)
print(f"Without X-API-Key: HTTP {no_auth_response.status_code}")
print(f"  Body: {no_auth_response.text}")
assert no_auth_response.status_code == 401, "Expected 401 without API key"
print("  ✓ Auth rejected as expected")

In [ ]:
# 4b + 4c — Correct API key; two messages

HEADERS = {
    "Content-Type": "application/json",
    "A2A-Version": "1.0",
    "X-API-Key": "test-key-123",
}

task_ids = []
for i, text in enumerate(["first task", "second task"]):
    response = httpx.post(
        "http://127.0.0.1:9998/",
        headers=HEADERS,
        json={
            "jsonrpc": "2.0", "id": f"r{i+1}", "method": "SendMessage",
            "params": {"message": {
                "message_id": f"m{i+1}", "role": "ROLE_USER",
                "parts": [{"text": text}],
            }},
        },
        timeout=10.0,
    )
    body = response.json()
    assert response.status_code == 200, f"Expected 200, got {response.status_code}"
    assert "error" not in body, f"Unexpected error: {body.get('error')}"
    task = body["result"]["task"]
    task_ids.append(task["id"])
    print(f"  ✓ Task {i+1}: id={task['id'][:8]}..., state={task['status']['state']}")
    artifact_text = task["artifacts"][0]["parts"][0]["text"]
    print(f"      artifact: '{artifact_text}'")

print(f"\nCaptured task IDs for persistence test in Step 5: {[tid[:8]+'...' for tid in task_ids]}")

## Step 5 — Direct SQLite inspection

The `DatabaseTaskStore` writes to a SQLAlchemy-managed SQLite file. Let's open it directly (without going through the SDK) and confirm the tasks from Step 4 are persisted.

This is also a sanity check on the SDK's schema choices — useful when planning migration strategies for production deployments.

In [ ]:
# Open the SQLite file directly — no SDK involvement, just sqlite3 stdlib
assert DB_PATH.exists(), f"DB file missing: {DB_PATH}"

conn = sqlite3.connect(str(DB_PATH))
cursor = conn.cursor()

# What tables exist?
cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='table'")
for table_name, ddl in cursor.fetchall():
    print(f"Table: {table_name}")
    print(f"  DDL: {ddl[:200]}...")

print()

# How many tasks?
cursor.execute("SELECT COUNT(*) FROM tasks")
n_tasks = cursor.fetchone()[0]
print(f"Tasks persisted: {n_tasks}")

# Show their IDs and states
cursor.execute("SELECT id, context_id FROM tasks")
for row in cursor.fetchall():
    print(f"  id={row[0][:8]}..., context={row[1][:8]}...")

# Confirm our Step 4 IDs are present
cursor.execute("SELECT id FROM tasks")
stored_ids = {row[0] for row in cursor.fetchall()}
missing = [tid for tid in task_ids if tid not in stored_ids]
assert not missing, f"Tasks missing from DB: {missing}"
print(f"\n✓ All {len(task_ids)} task IDs from Step 4 are in the SQLite store")

conn.close()

## Step 6 — Server restart; retrieve task via GetTask

The point of `DatabaseTaskStore` over `InMemoryTaskStore` is durability — tasks survive process restarts. Test this directly:

6a. Terminate the current server subprocess.
6b. Spawn a fresh subprocess (same code, same DB file, completely new process).
6c. Call `GetTask` with one of the task IDs from Step 4 — get back the persisted task.

In [ ]:
# 6a — terminate current server
server_process.terminate()
server_process.wait(timeout=5.0)
print(f"✓ Terminated original server (exit={server_process.returncode})")

# 6b — spawn fresh server (reads the same SQLite file)
server_process_v2 = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(2.0)

# Health probe
for _attempt in range(5):
    try:
        r = httpx.get("http://127.0.0.1:9998/.well-known/agent-card.json", timeout=2.0)
        if r.status_code == 200:
            break
    except (httpx.ConnectError, httpx.ReadTimeout):
        time.sleep(0.5)
else:
    if server_process_v2.poll() is not None:
        print(server_process_v2.stderr.read().decode()[:1500])
    raise RuntimeError("Restarted server failed to start")

print(f"✓ Restarted server (PID {server_process_v2.pid})")

In [ ]:
# 6c — GetTask via JSON-RPC; field name is `id` (not `name` — the v0.3 alias)

target_task_id = task_ids[0]
print(f"Requesting task: {target_task_id}")

get_response = httpx.post(
    "http://127.0.0.1:9998/",
    headers=HEADERS,
    json={
        "jsonrpc": "2.0", "id": "get-1", "method": "GetTask",
        "params": {"id": target_task_id},
    },
    timeout=5.0,
)
body = get_response.json()
if "error" in body:
    print(f"✗ Error: {body['error']}")
else:
    task = body["result"]
    print("\n✓ Retrieved persisted task across server restart:")
    print(f"  id: {task['id']}")
    print(f"  contextId: {task['contextId']}")
    print(f"  state: {task['status']['state']}")
    if "artifacts" in task and task["artifacts"]:
        print(f"  artifact: {task['artifacts'][0]['parts'][0]['text']}")

## Step 7 — Streaming `SendStreamingMessage` over SSE

The card already declares `streaming: true`. The dispatcher routes `SendStreamingMessage` calls to the SSE response handler — same `AgentExecutor.execute()` runs, but each event published via `TaskUpdater` flows back as a separate SSE event.

For a simple agent like ours, the canonical event sequence is four events:

1. Initial Task (state `SUBMITTED`)
2. `TaskStatusUpdateEvent` (state `WORKING`)
3. `TaskArtifactUpdateEvent` (the response artifact)
4. `TaskStatusUpdateEvent` (state `COMPLETED`)

httpx's `client.stream(...)` lets you iterate SSE lines as they arrive.

In [ ]:
async def consume_stream():
    """Consume SSE events from SendStreamingMessage one by one."""
    events_received = []

    async with httpx.AsyncClient() as client, client.stream(
        "POST",
        "http://127.0.0.1:9998/",
        headers=HEADERS,
        json={
            "jsonrpc": "2.0", "id": "stream-1", "method": "SendStreamingMessage",
            "params": {"message": {
                "message_id": "ms1", "role": "ROLE_USER",
                "parts": [{"text": "stream this"}],
            }},
        },
        timeout=15.0,
    ) as response:
        async for line in response.aiter_lines():
            if line.startswith("data: "):
                event_json = json.loads(line[6:])
                events_received.append(event_json)

    return events_received

events = await consume_stream()

print(f"✓ Received {len(events)} SSE events from the streaming endpoint\n")
for i, e in enumerate(events):
    result = e.get("result", {})
    # The result wrapper varies by event type
    if "task" in result:
        ev_type = f"initial Task (state={result['task']['status']['state']})"
    elif "statusUpdate" in result:
        ev_type = f"statusUpdate (state={result['statusUpdate']['status']['state']})"
    elif "artifactUpdate" in result:
        artifact = result["artifactUpdate"]["artifact"]
        artifact_text = artifact["parts"][0]["text"]
        ev_type = f"artifactUpdate (name='{artifact['name']}', text='{artifact_text}')"
    else:
        ev_type = f"unknown ({list(result.keys())})"
    print(f"  Event {i+1}: {ev_type}")

## Step 8 — OpenTelemetry spans from the request lifecycle

The server's OTel `SimpleSpanProcessor` has been writing spans to a JSON-line file throughout Steps 4-7. Each line is one finished span. Time to read what's there and see the request fanout.

For a single `SendMessage` request, expect ~40 spans across ~15 unique names — the heaviest fanout comes from the EventQueue's enqueue/dequeue operations as the agent publishes lifecycle events. The top of the stack is `JsonRpcDispatcher.handle_requests`; one level down is `DefaultRequestHandlerV2.on_message_send`; below that the executor fans out through the event queue and back.

In [ ]:
import contextlib
# Terminate the server so the SimpleSpanProcessor's final spans flush to disk
server_process_v2.terminate()
server_process_v2.wait(timeout=5.0)
print(f"✓ Stopped server (exit={server_process_v2.returncode})\n")
time.sleep(0.5)

# Read the spans
from collections import Counter

assert SPANS_PATH.exists(), f"Spans file missing: {SPANS_PATH}"
spans = []
for line in SPANS_PATH.read_text().splitlines():
    if line.strip():
        with contextlib.suppress(json.JSONDecodeError):
            spans.append(json.loads(line))

print(f"Total spans captured across Steps 4-7: {len(spans)}")
print(f"Unique span names: {len({s['name'] for s in spans})}\n")

counter = Counter(s["name"] for s in spans)
print("Top 12 span names by count:")
for name, count in counter.most_common(12):
    # Strip the long a2a.server prefix for readability
    short_name = name.replace("a2a.server.", "")
    print(f"  {count:>4} × {short_name}")

# Show unique trace IDs to demonstrate trace correlation
unique_traces = {s["trace_id"] for s in spans}
print(f"\n{len(unique_traces)} distinct trace ID(s) — each trace correlates one request's spans")

## What you've built and what's still ahead

You now have:

- 📄 **`production_agent_server.py`** (~120 lines) — a production-shape A2A endpoint
- 📄 **`signed_card.json`** — a real JWS-RS256-signed Agent Card with verified signature
- 📄 **`a2a_tasks.db`** — a SQLite database showing actual persisted task state
- 📄 **`a2a_spans.jsonl`** — OTel spans captured across all requests during the lab

Five of Module 6's six production concerns demonstrated end-to-end. The sixth (push notifications) was discussed but not implemented — it requires a webhook receiver on the client side and naturally belongs to Module 7's orchestrator-pattern lab where push notifications make sense in context (long-running cross-agent delegations).

## What's still ahead

- **Push notifications + OAuth2** — Module 7's lab will use a real auth issuer and demonstrate cross-agent webhook callbacks
- **PostgreSQL for the task store** — same `DatabaseTaskStore` API; change the connection string. Production patterns for connection pooling and migration strategies will land in adjacent path-08 content
- **OTel collector wiring** — the `SimpleSpanProcessor` + JSON file pattern here is for notebook inspection. Production uses `BatchSpanProcessor` exporting to Jaeger / Honeycomb / Datadog via OTLP
- **Module 7 — MCP + A2A composition** — the orchestrator pattern using `A2ACardResolver` + `ClientFactory` to discover and call remote agents; agents using MCP for their tools while using A2A to coordinate; the canonical hybrid pattern per Pattern 12

## Test yourself

Take the [A2A endpoint production-depth quiz](../../quizzes/foundations/a2a-endpoint-production-depth.md) — 8 questions covering Module 6 and this lab.